In [9]:
# ===============================
# 1. Import Libraries
# ===============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\BHUMIKA
[nltk_data]     RAVI\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
# ===============================
# 2. Load Dataset
# ===============================
df = pd.read_csv(r"C:\Users\BHUMIKA RAVI\Downloads\sentimentdataset.csv") 

# Drop unwanted columns (if present)
df = df.drop(columns=["Unnamed: 0", "Unnamed: 0.1"], errors='ignore')

# Preview dataset
print(df.head())

# Check structure
print(df.info())

# Check column names
print(df.columns)

                                                Text    Sentiment  \
0   Enjoying a beautiful day at the park!        ...   Positive     
1   Traffic was terrible this morning.           ...   Negative     
2   Just finished an amazing workout! 💪          ...   Positive     
3   Excited about the upcoming weekend getaway!  ...   Positive     
4   Trying out a new recipe for dinner tonight.  ...   Neutral      

             Timestamp            User     Platform  \
0  2023-01-15 12:30:00   User123          Twitter     
1  2023-01-15 08:45:00   CommuterX        Twitter     
2  2023-01-15 15:45:00   FitnessFan      Instagram    
3  2023-01-15 18:20:00   AdventureX       Facebook    
4  2023-01-15 19:55:00   ChefCook        Instagram    

                                     Hashtags  Retweets  Likes       Country  \
0   #Nature #Park                                  15.0   30.0     USA         
1   #Traffic #Morning                               5.0   10.0     Canada      
2   #Fitness #

In [11]:
# ===============================
# 3. Data Cleaning
# ===============================

# Remove missing values
df.dropna(inplace=True)

# Reset index after dropping rows
df.reset_index(drop=True, inplace=True)

# Check again
print(df.isnull().sum())
print("Shape after cleaning:", df.shape)

Text         0
Sentiment    0
Timestamp    0
User         0
Platform     0
Hashtags     0
Retweets     0
Likes        0
Country      0
Year         0
Month        0
Day          0
Hour         0
dtype: int64
Shape after cleaning: (732, 13)


In [12]:
# Function to clean text
def clean_text(text):
    text = text.lower()                        # Convert to lowercase
    text = re.sub(r"http\S+", "", text)        # Remove URLs
    text = re.sub(r"[^a-zA-Z\s]", "", text)    # Remove punctuation & numbers
    text = re.sub(r"\s+", " ", text).strip()   # Remove extra spaces
    return text

# Apply cleaning
df['cleaned_text'] = df['Text'].apply(clean_text)

# Display examples
print("Original:", df['Text'].iloc[0])
print("Cleaned :", df['cleaned_text'].iloc[0])

Original:  Enjoying a beautiful day at the park!              
Cleaned : enjoying a beautiful day at the park


In [13]:
# ===============================
# 4. Text Preprocessing
# ===============================
ps = PorterStemmer()
corpus = []

for i in range(len(df)):
    
    # Remove special characters
    text = re.sub('[^a-zA-Z]', ' ', df['Text'][i])
    
    # Convert to lowercase
    text = text.lower()
    
    # Tokenization
    text = text.split()
    
    # Remove stopwords + stemming
    text = [ps.stem(word) for word in text if word not in stopwords.words('english')]
    
    # Join back to sentence
    text = ' '.join(text)
    
    corpus.append(text)

# Preview processed text
print(corpus[:5])

['enjoy beauti day park', 'traffic terribl morn', 'finish amaz workout', 'excit upcom weekend getaway', 'tri new recip dinner tonight']


In [14]:
# ===============================
# 5. Feature Extraction (TF-IDF)
# ===============================
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000)


# Convert text data into numerical form
X = tfidf.fit_transform(corpus).toarray()

# Target variable
y = df['Sentiment']

# Check shapes
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (732, 1938)
y shape: (732,)


In [15]:
# ===============================
# 6. Train-Test Split
# ===============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Check shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (585, 1938)
X_test: (147, 1938)
y_train: (585,)
y_test: (147,)


In [16]:
# ===============================
# 7. Model Training
# ===============================
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()

# Train the model
model.fit(X_train, y_train)

LogisticRegression()

In [23]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Load dataset
df = pd.read_csv(r"C:\Users\BHUMIKA RAVI\Downloads\sentimentdataset.csv") 

df = df[['Text', 'Sentiment']].dropna().reset_index(drop=True)

# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['cleaned_text'] = df['Text'].apply(clean_text)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_text'], df['Sentiment'], test_size=0.2, random_state=42)

# Create and fit vectorizer
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# ---- Now prediction works ----
sample_post = "I really love this product!"
sample_clean = clean_text(sample_post)
pred = model.predict(vectorizer.transform([sample_clean]))[0]

if pred == "Positive":
    sentiment = "Positive"
elif pred == "Negative":
    sentiment = "Negative"
else:
    sentiment = "Neutral"

print("Predicted Sentiment:", sentiment)

Predicted Sentiment: Neutral


In [24]:
# ===============================
# 9. Model Evaluation
# ===============================

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

NameError: name 'y_pred' is not defined